# 🌿 Leaf Disease Classification — Custom CNN (1 Fase, 65 Epoch) — Split 50/50 Train-Test

Model dibangun dari nol (bukan transfer learning), sehingga training hanya **1 fase** — tidak ada freeze/unfreeze base model karena tidak ada backbone pretrained.

> **📝 Revisi (perbaikan instabilitas training):**
> Run sebelumnya berhenti sendiri (di-interrupt) di epoch 6 karena `val_loss` melonjak drastis (3.8 → 10.68) dan `val_accuracy` jatuh (0.298 → 0.092) di epoch 5, padahal `train_accuracy` terus naik — tanda klasik LR terlalu tinggi / gradien meledak. Split tetap 50/50 train-test tanpa folder valid terpisah (`test_ds` dipakai sebagai validation, sesuai desain eksperimen). Perbaikan yang diterapkan di notebook ini:
> 1. **Gradient clipping (`clipnorm=1.0`)** pada optimizer — mencegah lonjakan gradien yang memicu val_loss meledak.
> 2. **Peak LR diturunkan 1e-3 → 3e-4** & **warmup diperpanjang 5 → 8 epoch** — kenaikan LR lebih landai.
> 3. **Augmentasi warna diperlembut** — `random_hue`/`random_saturation` dihapus (warna adalah sinyal diagnostik penyakit daun), brightness/contrast diperlemah.
> 4. **Monitor EarlyStopping/ModelCheckpoint diganti ke `val_loss`** (lebih stabil dari `val_accuracy` untuk 72 kelas).
> 5. **L2 regularization ditambahkan ke semua conv block**, sebelumnya hanya ada di Dense layer.


## 1. Import & Konfigurasi GPU

In [1]:
import os, json, random, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.utils import class_weight
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.manifold import TSNE

import tensorflow as tf
from tensorflow.keras import layers, models, regularizers, callbacks

print("TensorFlow :", tf.__version__)
print("Keras      :", tf.keras.__version__)


c:\Users\crism\AppData\Local\Programs\Python\Python310\lib\site-packages\google\api_core\_python_version_support.py:255: FutureWarning: You are using a Python version (3.10.0) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


TensorFlow : 2.10.1
Keras      : 2.10.0


In [2]:
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    tf.config.set_visible_devices(gpus[0], 'GPU')
    print(f"✅ GPU: {gpus[0].name}")
else:
    print("⚠️  CPU mode")

print("Logical devices:", tf.config.list_logical_devices())
print("CUDA built     :", tf.test.is_built_with_cuda())


✅ GPU: /physical_device:GPU:0
Logical devices: [LogicalDevice(name='/device:CPU:0', device_type='CPU'), LogicalDevice(name='/device:GPU:0', device_type='GPU')]
CUDA built     : True


## 2. Konfigurasi Path & Parameter

In [3]:
BASE_TRAIN = r"C:\Users\crism\Music\50 50\train"
BASE_TEST  = r"C:\Users\crism\Music\50 50\test"
SAVE_DIR   = r"C:\Users\crism\Music\New folder (5)\final_output\(32) Custom CNN 5050"
SAVE_MODEL = os.path.join(SAVE_DIR, "models")
SAVE_LABEL = os.path.join(SAVE_DIR, "labels")
os.makedirs(SAVE_MODEL, exist_ok=True)
os.makedirs(SAVE_LABEL, exist_ok=True)

TRAIN_DIR = BASE_TRAIN
TEST_DIR  = BASE_TEST
# Tidak ada folder valid terpisah — test_ds dipakai langsung sebagai validation_data saat training

IMG_SIZE    = (256, 256)
IMG_SHAPE   = (256, 256, 3)
NUM_CLASSES = 72

BATCH_SIZE = 32
AUTOTUNE   = tf.data.AUTOTUNE

SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

print(f"BATCH_SIZE : {BATCH_SIZE}")
print(f"IMG_SIZE   : {IMG_SIZE}")
for name, path in [("train", TRAIN_DIR), ("test", TEST_DIR)]:
    ok = "✅" if os.path.isdir(path) else "❌"
    print(f"  {ok} {name}: {path}")


BATCH_SIZE : 32
IMG_SIZE   : (256, 256)
  ✅ train: C:\Users\crism\Music\50 50\train
  ✅ test: C:\Users\crism\Music\50 50\test


## 3. tf.data Pipeline

**Catatan penting:** karena model ini dibangun dari nol (tidak pakai backbone pretrained seperti MobileNetV2/InceptionV3), preprocessing gambar cukup di-normalisasi ke rentang `[0,1]` (`Rescaling(1./255)`), **bukan** `preprocess_input()` khusus arsitektur tertentu.

**Catatan split 50/50:** tidak ada folder `valid` terpisah — `test_ds` dipakai juga sebagai `validation_data` saat training (sama seperti notebook InceptionV3/MobileNetV2 versi 50/50 sebelumnya). Karena itu, stabilitas training (LR & gradient clipping di Bagian 5) jadi lebih penting daripada biasanya — tidak ada validation set independen yang bisa "menyelamatkan" pemilihan model kalau training sempat tidak stabil.


In [4]:
rotation_layer = tf.keras.layers.RandomRotation(
    factor=0.1,           # ±36 derajat  (0.1 × 360°)
    fill_mode="reflect",
    seed=SEED
)

def load_image(path, label):
    raw   = tf.io.read_file(path)
    image = tf.image.decode_jpeg(raw, channels=3)
    image = tf.cast(image, tf.float32)
    image = tf.image.resize(image, IMG_SIZE)
    return image, label

@tf.function
def augment_train(image, label):
    """
    Augmentasi on-the-fly.
    Input  : float32 [0,255]
    Output : dinormalisasi ke [0,1]

    CATATAN PERBAIKAN:
    - random_hue & random_saturation DIHAPUS. Untuk klasifikasi penyakit daun,
      warna (klorosis/nekrosis/bercak) adalah sinyal diagnostik utama — hue/saturation
      jitter yang terlalu agresif bisa merusak justru fitur yang ingin dipelajari model.
    - brightness & contrast tetap dipakai tapi diperlemah sedikit supaya augmentasi
      tidak terlalu jauh mengubah distribusi warna asli.
    """
    image = tf.image.random_flip_left_right(image)
    image = tf.image.random_flip_up_down(image)
    image = tf.image.random_brightness(image, max_delta=0.12 * 255)
    image = tf.image.random_contrast(image, lower=0.9, upper=1.1)

    crop_frac = tf.random.uniform([], 0.9, 1.0)
    crop_h    = tf.cast(tf.cast(IMG_SIZE[0], tf.float32) * crop_frac, tf.int32)
    crop_w    = tf.cast(tf.cast(IMG_SIZE[1], tf.float32) * crop_frac, tf.int32)
    image     = tf.image.random_crop(image, size=[crop_h, crop_w, 3])
    image     = tf.image.resize(image, IMG_SIZE)

    image = rotation_layer(tf.expand_dims(image, 0), training=True)[0]

    image = tf.clip_by_value(image, 0., 255.)
    image = image / 255.0                     # ✅ normalisasi custom CNN, bukan preprocess_input()
    return image, label

@tf.function
def preprocess_eval(image, label):
    image = image / 255.0
    return image, label

def one_hot(image, label):
    return image, tf.one_hot(label, NUM_CLASSES)

_raw_train = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR, image_size=IMG_SIZE, batch_size=None, shuffle=True,
    seed=SEED, label_mode="int"
)
class_names   = _raw_train.class_names
class_indices = {name: i for i, name in enumerate(class_names)}
print(f"Jumlah kelas terdeteksi: {len(class_names)}")
assert len(class_names) == NUM_CLASSES, \
    f"Kelas terdeteksi {len(class_names)} ≠ NUM_CLASSES={NUM_CLASSES}"

train_labels = []
for _, lbl in _raw_train:
    train_labels.append(lbl.numpy())
train_labels = np.array(train_labels)
print(f"Total sampel train : {len(train_labels)}")

train_ds = (
    _raw_train
    .map(lambda img, lbl: (tf.cast(img, tf.float32), lbl), num_parallel_calls=AUTOTUNE)
    .map(augment_train, num_parallel_calls=AUTOTUNE)
    .map(one_hot,       num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

_raw_test = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR, image_size=IMG_SIZE, batch_size=None, shuffle=False,
    seed=SEED, label_mode="int"
)
test_ds = (
    _raw_test
    .map(lambda img, lbl: (tf.cast(img, tf.float32), lbl), num_parallel_calls=AUTOTUNE)
    .map(preprocess_eval, num_parallel_calls=AUTOTUNE)
    .map(one_hot,         num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

# Sesuai desain eksperimen: split hanya train/test 50/50, tidak ada folder valid terpisah.
# test_ds dipakai juga sebagai valid_ds (konsisten dengan notebook-notebook split lain).
valid_ds = test_ds

print("\n✅ Semua pipeline siap (tanpa folder valid — test_ds dipakai sebagai validation).")
print(f"  train_ds : {train_ds}")
print(f"  test_ds  : {test_ds}  (juga dipakai sebagai valid_ds)")


Found 36206 files belonging to 72 classes.
Jumlah kelas terdeteksi: 72
Total sampel train : 36206
Found 35794 files belonging to 72 classes.

✅ Semua pipeline siap (tanpa folder valid — test_ds dipakai sebagai validation).
  train_ds : <PrefetchDataset element_spec=(TensorSpec(shape=(None, 256, 256, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None, 72), dtype=tf.float32, name=None))>
  test_ds  : <PrefetchDataset element_spec=(TensorSpec(shape=(None, 256, 256, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None, 72), dtype=tf.float32, name=None))>  (juga dipakai sebagai valid_ds)


## 4. Class Weights (Imbalance Handling)

In [5]:
unique_labels = np.unique(train_labels)
calc_weights  = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=unique_labels,
    y=train_labels
)
class_weights_dict = {int(k): float(min(v, 5.0)) for k, v in zip(unique_labels, calc_weights)}
print(f"Class weights ({len(class_weights_dict)} kelas) — clipped at 5.0")
w_vals = list(class_weights_dict.values())
print(f"  min={min(w_vals):.4f} | max={max(w_vals):.4f} | mean={np.mean(w_vals):.4f}")


Class weights (72 kelas) — clipped at 5.0
  min=0.7123 | max=1.0057 | mean=1.0016


## 5. Arsitektur Model — Custom CNN dari Nol

5 blok Conv (32→64→128→256→512), tiap blok: `Conv2D → BatchNorm → ReLU → Conv2D → BatchNorm → ReLU → MaxPool → Dropout`. Ditutup `GlobalAveragePooling2D` + Dense head, supaya jumlah parameter di FC layer tetap kecil (menghindari overfitting parah khas CNN from-scratch). `label_smoothing=0.02` dan `L2=1e-5` dipakai dari awal (bukan `0.1`/`1e-4`) supaya val_loss tidak kejebak floor tinggi seperti yang terjadi di notebook-notebook sebelumnya.

In [6]:
strategy = tf.distribute.MirroredStrategy()
print(f"Jumlah device: {strategy.num_replicas_in_sync}")

# ============================================================
# PERBAIKAN: tambahkan gradient clipping (clipnorm=1.0).
# Ini adalah penyebab utama val_loss "meledak" (val_loss: 3.8 -> 10.68)
# dan val_accuracy jatuh drastis (0.298 -> 0.092) di epoch 5 pada run
# sebelumnya: begitu learning rate warmup naik mendekati peak (1e-3),
# beberapa batch menghasilkan gradien besar yang membuat bobot ter-update
# terlalu jauh sekaligus. clipnorm membatasi besar gradien per step
# tanpa mengubah arahnya, jadi update tetap stabil.
# ============================================================
try:
    from tensorflow.keras.optimizers.experimental import AdamW
    optimizer = AdamW(learning_rate=1e-3, weight_decay=1e-4, clipnorm=1.0)
    print("Optimizer : AdamW (experimental, clipnorm=1.0)")
except Exception:
    optimizer = tf.keras.optimizers.Adam(learning_rate=1e-3, clipnorm=1.0)
    print("Optimizer : Adam (clipnorm=1.0)")


CONV_L2 = regularizers.l2(1e-5)

def conv_block(x, filters, block_id, pool=True, dropout=0.0):
    x = layers.Conv2D(filters, 3, padding="same", use_bias=False,
                       kernel_regularizer=CONV_L2,
                       name=f"block{block_id}_conv1")(x)
    x = layers.BatchNormalization(name=f"block{block_id}_bn1")(x)
    x = layers.Activation("relu", name=f"block{block_id}_relu1")(x)

    x = layers.Conv2D(filters, 3, padding="same", use_bias=False,
                       kernel_regularizer=CONV_L2,
                       name=f"block{block_id}_conv2")(x)
    x = layers.BatchNormalization(name=f"block{block_id}_bn2")(x)
    x = layers.Activation("relu", name=f"block{block_id}_relu2")(x)

    if pool:
        x = layers.MaxPooling2D(2, name=f"block{block_id}_pool")(x)
    if dropout > 0:
        x = layers.Dropout(dropout, name=f"block{block_id}_drop")(x)
    return x


with strategy.scope():

    def build_model(num_classes):
        inputs = layers.Input(shape=IMG_SHAPE, name="input")

        x = conv_block(inputs, 32,  block_id=1, dropout=0.20)   # 256 -> 128
        x = conv_block(x,      64,  block_id=2, dropout=0.25)   # 128 -> 64
        x = conv_block(x,      128, block_id=3, dropout=0.30)   #  64 -> 32
        x = conv_block(x,      256, block_id=4, dropout=0.35)   #  32 -> 16
        x = conv_block(x,      512, block_id=5, dropout=0.40,   #  16 -> 8
                        pool=True)
        x = layers.Conv2D(512, 3, padding="same", activation="relu",
                           kernel_regularizer=CONV_L2,
                           name="last_conv")(x)                 # dipakai Grad-CAM

        x = layers.GlobalAveragePooling2D(name="gap")(x)

        x = layers.Dense(
            512,
            activation="relu",
            kernel_regularizer=regularizers.l2(1e-5),
            name="feature_dense"
        )(x)
        x = layers.BatchNormalization()(x)
        x = layers.Dropout(0.40)(x)

        outputs = layers.Dense(num_classes, activation="softmax", name="predictions")(x)

        return models.Model(inputs, outputs, name="custom_cnn")

    model = build_model(NUM_CLASSES)

    loss_fn = tf.keras.losses.CategoricalCrossentropy(
        label_smoothing=0.02   # floor loss teoritis ~0.18 utk 72 kelas, bukan 0.1 (~0.74)
    )

    model.compile(
        optimizer=optimizer,
        loss=loss_fn,
        metrics=[tf.keras.metrics.CategoricalAccuracy(name="accuracy")]
    )

trainable    = np.sum([np.prod(v.shape) for v in model.trainable_weights])
nontrainable = np.sum([np.prod(v.shape) for v in model.non_trainable_weights])

print("=" * 60)
print(f"Total trainable params     : {trainable:,}")
print(f"Total non-trainable params : {nontrainable:,}")
print("=" * 60)

model.summary()


INFO:tensorflow:Using MirroredStrategy with devices ('/job:localhost/replica:0/task:0/device:GPU:0',)
Jumlah device: 1
Optimizer : AdamW (experimental, clipnorm=1.0)
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:

## 6. Cosine Annealing LR dengan Warmup

In [7]:
class WarmupCosineDecay(callbacks.Callback):
    """LR naik linear saat warmup, lalu turun cosine hingga min_lr."""
    def __init__(self, total_epochs, warmup_epochs=5,
                 start_lr=1e-5, peak_lr=1e-3, min_lr=1e-6):
        super().__init__()
        self.total_epochs  = total_epochs
        self.warmup_epochs = warmup_epochs
        self.start_lr      = start_lr
        self.peak_lr       = peak_lr
        self.min_lr        = min_lr

    def on_epoch_begin(self, epoch, logs=None):
        if epoch < self.warmup_epochs:
            lr = self.start_lr + (self.peak_lr - self.start_lr) * (epoch / self.warmup_epochs)
        else:
            progress = (epoch - self.warmup_epochs) / (self.total_epochs - self.warmup_epochs)
            lr = self.min_lr + 0.5 * (self.peak_lr - self.min_lr) * (
                1 + math.cos(math.pi * progress))
        tf.keras.backend.set_value(self.model.optimizer.learning_rate, lr)
        print(f"  [LR] Epoch {epoch+1}: lr = {lr:.2e}")

print("WarmupCosineDecay callback siap.")


WarmupCosineDecay callback siap.


## 7. Callbacks — 1 Fase, 65 Epoch

Karena training dari nol (bukan fine-tuning), `peak_lr` awalnya dibuat lebih tinggi dibanding notebook transfer-learning — namun ini justru memicu instabilitas (lihat catatan revisi di bagian atas notebook). Setelah revisi: `peak_lr=3e-4`, `warmup_epochs=8`, gradient clipping aktif di optimizer, dan monitor `EarlyStopping`/`ModelCheckpoint` memakai `val_loss` dengan `patience=18` (CNN from-scratch butuh lebih banyak epoch untuk konvergen dibanding fine-tuning model pretrained).


In [8]:
EPOCHS = 65

# ============================================================
# PERBAIKAN:
# - peak_lr diturunkan 1e-3 -> 3e-4. 1e-3 terlalu tinggi untuk CNN
#   5-blok yang dalam + augmentasi cukup berat; dikombinasikan dengan
#   BatchNorm, ini yang memicu lonjakan val_loss di epoch 5 (run sebelumnya).
# - warmup_epochs diperpanjang 5 -> 8 supaya kenaikan LR lebih landai.
# - monitor diganti ke val_loss (bukan val_accuracy) untuk EarlyStopping
#   & ModelCheckpoint. val_loss lebih sensitif & stabil sebagai sinyal
#   generalisasi untuk masalah 72 kelas, sementara val_accuracy bisa
#   terlihat "membaik" walau model sebenarnya mulai overfit/tidak stabil.
# - patience dinaikkan ke 18 karena peak_lr yang lebih rendah butuh
#   sedikit lebih banyak epoch untuk konvergen penuh.
# ============================================================
cb = [
    WarmupCosineDecay(
        total_epochs=EPOCHS,
        warmup_epochs=8,
        start_lr=1e-5,
        peak_lr=3e-4,
        min_lr=1e-6
    ),
    callbacks.EarlyStopping(
        monitor="val_loss",
        mode="min",
        patience=18,
        restore_best_weights=True,
        verbose=1
    ),
    callbacks.ModelCheckpoint(
        os.path.join(SAVE_MODEL, "custom_cnn_best.keras"),
        monitor="val_loss",
        mode="min",
        save_best_only=True,
        verbose=1
    )
]

print("Callbacks siap.")
print(f"  Total epoch : {EPOCHS}")
print(f"  LR peak     : 3e-4 | warmup 8 epoch | EarlyStopping(val_loss) patience 18")


Callbacks siap.
  Total epoch : 65
  LR peak     : 3e-4 | warmup 8 epoch | EarlyStopping(val_loss) patience 18


## 8. Training — Satu Fase Penuh

In [9]:
print("=" * 60)
print(f"TRAINING CUSTOM CNN — 1 FASE, {EPOCHS} EPOCH")
print("=" * 60)

if tf.config.list_physical_devices('GPU'):
    print(f"✅ GPU: {tf.config.list_physical_devices('GPU')[0].name}")

history = model.fit(
    train_ds,
    validation_data=valid_ds,
    epochs=EPOCHS,
    callbacks=cb,
    class_weight=class_weights_dict
)


TRAINING CUSTOM CNN — 1 FASE, 65 EPOCH
✅ GPU: /physical_device:GPU:0
  [LR] Epoch 1: lr = 1.00e-05
Epoch 1/65
1132/1132 [==============================] - ETA: 0s - loss: 3.5325 - accuracy: 0.1415
Epoch 1: val_loss improved from inf to 6.13214, saving model to C:\Users\crism\Music\New folder (5)\final_output\(32) Custom CNN 5050\models\custom_cnn_best.keras
1132/1132 [==============================] - 397s 332ms/step - loss: 3.5325 - accuracy: 0.1415 - val_loss: 6.1321 - val_accuracy: 0.0328
  [LR] Epoch 2: lr = 4.62e-05
Epoch 2/65
1132/1132 [==============================] - ETA: 0s - loss: 2.6405 - accuracy: 0.3092
Epoch 2: val_loss improved from 6.13214 to 3.22365, saving model to C:\Users\crism\Music\New folder (5)\final_output\(32) Custom CNN 5050\models\custom_cnn_best.keras
1132/1132 [==============================] - 373s 330ms/step - loss: 2.6405 - accuracy: 0.3092 - val_loss: 3.2236 - val_accuracy: 0.2153
  [LR] Epoch 3: lr = 8.25e-05
Epoch 3/65
1132/1132 [===================

KeyboardInterrupt: 

## 9. Grafik Training

In [ ]:
def plot_history(history, title):
    acc      = history.history['accuracy']
    val_acc  = history.history['val_accuracy']
    loss     = history.history['loss']
    val_loss = history.history['val_loss']
    ep       = range(1, len(acc) + 1)

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle(title, fontsize=13, fontweight='bold')

    axes[0].plot(ep, acc,     'b-o', ms=3, label='Train')
    axes[0].plot(ep, val_acc, 'g-o', ms=3, label='Valid')
    best_e = int(np.argmax(val_acc)) + 1
    axes[0].axvline(best_e, color='r', linestyle='--', alpha=0.5, label=f'Best ep {best_e}')
    axes[0].set(title='Accuracy', xlabel='Epoch', ylabel='Accuracy', ylim=[0,1])
    axes[0].legend(); axes[0].grid(True, alpha=0.4)

    axes[1].plot(ep, loss,     'b-o', ms=3, label='Train')
    axes[1].plot(ep, val_loss, 'g-o', ms=3, label='Valid')
    axes[1].set(title='Loss', xlabel='Epoch', ylabel='Loss')
    axes[1].legend(); axes[1].grid(True, alpha=0.4)

    gap = [a - v for a, v in zip(acc, val_acc)]
    axes[2].plot(ep, gap, 'r-o', ms=3)
    axes[2].axhline(0, color='k', linestyle='--', alpha=0.5)
    axes[2].fill_between(ep, gap, 0,
                         where=[g > 0 for g in gap], alpha=0.2, color='orange', label='Overfitting')
    axes[2].fill_between(ep, gap, 0,
                         where=[g < 0 for g in gap], alpha=0.2, color='blue', label='Underfitting')
    axes[2].set(title='Train-Val Gap', xlabel='Epoch', ylabel='train_acc - val_acc')
    axes[2].legend(); axes[2].grid(True, alpha=0.4)

    plt.tight_layout(); plt.show()
    print(f"Best val_acc : {max(val_acc):.4f} @ epoch {best_e}")
    print(f"Final gap    : {gap[-1]:+.4f} ({'overfit' if gap[-1] > 0.05 else 'underfit' if gap[-1] < -0.02 else 'OK'})")

plot_history(history, "Custom CNN — 1 Fase (65 Epoch)")


## 10. Evaluasi TEST SET

In [ ]:
print('=' * 60)
print('Evaluasi TEST SET')
print('=' * 60)

y_pred_probs = model.predict(test_ds, verbose=1)
y_pred       = np.argmax(y_pred_probs, axis=1)

y_true = []
for _, labels in test_ds:
    y_true.extend(np.argmax(labels.numpy(), axis=1))
y_true = np.array(y_true)

acc_test = np.mean(y_true == y_pred)
print(f'\nTest Accuracy: {acc_test:.4f}\n')

report = classification_report(y_true, y_pred, target_names=class_names, digits=4)
print(report)

os.makedirs(SAVE_DIR, exist_ok=True)

txt_path = os.path.join(SAVE_DIR, 'classification_report_custom_cnn.txt')
with open(txt_path, 'w', encoding='utf-8') as f:
    f.write('TEST SET - CUSTOM CNN (1 FASE)\n')
    f.write('=' * 60 + '\n\n')
    f.write(f'Accuracy : {acc_test:.4f}\n\n')
    f.write(report)

report_dict = classification_report(y_true, y_pred, target_names=class_names, output_dict=True)
df_report = pd.DataFrame(report_dict).transpose()
csv_path  = os.path.join(SAVE_DIR, 'classification_report_custom_cnn.csv')
df_report.to_csv(csv_path)

cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(12, 10))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
disp.plot(cmap='Blues', xticks_rotation=90, ax=ax, colorbar=False)
plt.title('Confusion Matrix — Custom CNN (1 Fase)')
plt.tight_layout()

cm_path = os.path.join(SAVE_DIR, 'confusion_matrix_custom_cnn.png')
plt.savefig(cm_path, dpi=300, bbox_inches='tight')
plt.show()

print('=' * 60)
print('SEMUA HASIL BERHASIL DISIMPAN')
print('=' * 60)
print('Accuracy         :', acc_test)
print('TXT Report       :', txt_path)
print('CSV Report       :', csv_path)
print('Confusion Matrix :', cm_path)


## 11. Grad-CAM — Helper & Visualisasi

Layer konvolusi terakhir diberi nama `last_conv` di arsitektur model, jadi Grad-CAM tinggal merujuk ke nama itu (beda dari notebook InceptionV3 yang pakai `mixed10`).

In [ ]:
def make_gradcam_heatmap(img_array, model, last_conv_layer_name="last_conv", pred_index=None):
    grad_model = tf.keras.models.Model(
        model.inputs, [model.get_layer(last_conv_layer_name).output, model.output]
    )
    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array)
        if pred_index is None:
            pred_index = tf.argmax(predictions[0])
        class_channel = predictions[:, pred_index]

    grads = tape.gradient(class_channel, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    conv_outputs = conv_outputs[0]
    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy()


def show_gradcam(image_path, model, last_conv_layer_name="last_conv"):
    img = tf.keras.utils.load_img(image_path, target_size=IMG_SIZE)
    img_array = tf.keras.utils.img_to_array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)

    preds = model.predict(img_array, verbose=0)
    pred_class = class_names[np.argmax(preds[0])]
    confidence = np.max(preds[0])

    heatmap = make_gradcam_heatmap(img_array, model, last_conv_layer_name)
    heatmap = np.uint8(255 * heatmap)

    import matplotlib.cm as cm
    jet = cm.get_cmap("jet")
    jet_heatmap = jet(np.arange(256))[:, :3][heatmap]
    jet_heatmap = tf.keras.utils.array_to_img(jet_heatmap)
    jet_heatmap = jet_heatmap.resize(IMG_SIZE)
    jet_heatmap = tf.keras.utils.img_to_array(jet_heatmap)

    superimposed = jet_heatmap * 0.4 + (img_array[0] * 255)
    superimposed = tf.keras.utils.array_to_img(superimposed)

    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    axes[0].imshow(img_array[0]); axes[0].set_title(f"Asli\nPred: {pred_class} ({confidence:.2%})")
    axes[0].axis("off")
    axes[1].imshow(superimposed); axes[1].set_title("Grad-CAM")
    axes[1].axis("off")
    plt.tight_layout(); plt.show()

print("Grad-CAM helper siap. Contoh pemakaian:")
print('  show_gradcam("path/ke/gambar.jpg", model)')


## 12. Simpan Model & Label

In [ ]:
save_path = os.path.join(SAVE_MODEL, "Leaf_Disease_CustomCNN_1Phase.keras")
model.save(save_path)
print(f"✅ Model disimpan : {save_path}")

label_out = os.path.join(SAVE_LABEL, "class_indices.json")
with open(label_out, "w", encoding="utf-8") as f:
    json.dump(class_indices, f, indent=2, ensure_ascii=False)

print("Ringkasan:")
print(f"  Model  : {save_path}")
print(f"  Label  : {label_out}")
